In [ ]:
!pip install langchain langchain-community chromadb groq langchain-groq pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 837.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All imports successful")

All imports successful


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "your_groq_key_here"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Test it is working
response = llm.invoke("What is a electricity grid in one sentence?")
print(response.content)

An electricity grid is a network of power plants, transmission lines, substations, and distribution lines that work together to transmit and distribute electricity from its source to consumers, ensuring a reliable and efficient supply of power.


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

# Public EIA page about electricity markets - free, no compliance issues
loader = WebBaseLoader(
    web_paths=["https://www.eia.gov/energyexplained/electricity/electricity-in-the-us.php"],
)

docs = loader.load()

print(f"Loaded {len(docs)} document")
print(f"Total characters: {len(docs[0].page_content)}")
print("\nFirst 500 characters:")
print(docs[0].page_content[:500])

Loaded 1 document
Total characters: 17952

First 500 characters:

























Electricity in the U.S. - U.S. Energy Information Administration (EIA)





























Skip to sub-navigation


U.S. Energy Information Administration - EIA - Independent Statistics and Analysis











Menu





					Sources & Uses



					Topics



					Geography



					Tools



					Education



					News












Sources & Uses


Topics


Geography


Tools


Education


News










Petroleum & Other Liquids

Crude oil, gasoline, heating o


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(docs)

print(f"Total chunks created: {len(splits)}")
print("\nExample chunk:")
print(splits[5].page_content)

Total chunks created: 50

Example chunk:
Analysis & Projections

Monthly and yearly energy forecasts, analysis of energy topics, financial analysis, congressional reports.
								


Short-Term Energy Outlook


Annual Energy Outlook


International Energy Outlook






Markets & Finance

Financial market analysis and financial data for major energy companies.


Market Prices and Uncertainty Report


Energy & Financial Markets: What Drives Crude Oil Prices?








Environment


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

print("Vector store created successfully")
print(f"Total vectors stored: {vectorstore._collection.count()}")

/tmp/ipykernel_2237/186910957.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully
Total vectors stored: 50


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

prompt = PromptTemplate.from_template("""
You are an expert energy market assistant at GridCore Systems.
Answer the question using only the context provided below.
If the answer is not in the context, say "I don't have enough information on that."
Be concise and technical.

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully")

RAG chain built successfully


In [ ]:
question = "How is electricity generated in the US?"
answer = rag_chain.invoke(question)
print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: How is electricity generated in the US?

Answer: Electricity in the US is generated from diverse energy sources and technologies, primarily categorized into three major groups: 

1. Fossil fuels (coal, natural gas, and petroleum)
2. Nuclear energy
3. Renewable energy

Specifically, most electricity is generated with steam turbines that use fossil fuels, nuclear, biomass, geothermal, or solar thermal energy. Other major electricity generation technologies include gas turbines, hydro (water) turbines, wind turbines, and solar photovoltaics.


In [ ]:
question = "What is the price of electricity in New York today?"
answer = rag_chain.invoke(question)
print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: What is the price of electricity in New York today?

Answer: I don't have enough information on that.


In [ ]:
urls = [
    "https://www.eia.gov/energyexplained/electricity/electricity-in-the-us.php",
    "https://www.eia.gov/energyexplained/electricity/electricity-in-the-us-generation-capacity-and-sales.php",
    "https://www.eia.gov/energyexplained/electricity/delivery-to-consumers.php",
]

all_docs = []

for url in urls:
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()
        all_docs.extend(docs)
        print(f"Loaded: {url}")
    except Exception as e:
        print(f"Failed: {url}, {e}")

print(f"\nTotal documents loaded: {len(all_docs)}")

splits = text_splitter.split_documents(all_docs)
print(f"Total chunks: {len(splits)}")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("\nKnowledge base updated successfully")

Loaded: https://www.eia.gov/energyexplained/electricity/electricity-in-the-us.php
Loaded: https://www.eia.gov/energyexplained/electricity/electricity-in-the-us-generation-capacity-and-sales.php
Loaded: https://www.eia.gov/energyexplained/electricity/delivery-to-consumers.php

Total documents loaded: 3
Total chunks: 197

Knowledge base updated successfully


In [ ]:
questions = [
    "How is electricity delivered to consumers?",
    "What is the difference between transmission and distribution?",
    "What factors affect electricity generation capacity?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-"*50)


Q: How is electricity delivered to consumers?
A: Electricity is delivered to consumers through transmission and distribution power lines, operated by local electric utilities.
--------------------------------------------------

Q: What is the difference between transmission and distribution?
A: Transmission and distribution refer to the stages of electricity delivery. 

- Transmission involves the high-voltage electricity carried over long distances through high-voltage transmission lines, primarily using step-up transformers at power plants to increase voltage for efficient transmission.
- Distribution involves the lower-voltage electricity delivered to customers through distribution transmission lines, primarily using step-down transformers at substations to reduce voltage for safe use in homes and businesses.
--------------------------------------------------

Q: What factors affect electricity generation capacity?
A: According to the context, I don't have enough information on the

In [ ]:
def ask_energy_agent(question):
    answer = rag_chain.invoke(question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-"*50)

print("ENERGY MARKET INTELLIGENCE ASSISTANT")
print("GridCore Systems - Internal Knowledge Agent")
print("="*50)

ask_energy_agent("How is electricity generated in the US?")
ask_energy_agent("What is the role of substations in electricity delivery?")
ask_energy_agent("What renewable energy sources are used for electricity?")

ENERGY MARKET INTELLIGENCE ASSISTANT
GridCore Systems - Internal Knowledge Agent
Q: How is electricity generated in the US?
A: Electricity in the US is produced from diverse energy sources and technologies, including various combinations of:

1. Fossil fuels (coal, natural gas, and oil)
2. Nuclear power
3. Renewable energy sources (solar, wind, hydroelectric, geothermal, and biomass)
--------------------------------------------------
Q: What is the role of substations in electricity delivery?
A: Substations play a crucial role in adjusting voltage levels through transformers, enabling the efficient transmission and distribution of electricity from power plants to customers via transmission and distribution power lines.
--------------------------------------------------
Q: What renewable energy sources are used for electricity?
A: I don't have enough information on that.
--------------------------------------------------
